# Connecting Week 2 and Week 3: Fine-Tuning the Minimal VLM on Real Document Extraction

Week 2 built a minimal VLM (`minimal_vlm_llava_stage2_lora.ipynb`, Stage 2: frozen CLIP `openai/clip-vit-base-patch32` + LoRA-adapted `SmolLM2-135M` + a warm-started projector) and evaluated it on Flickr8k captioning. Week 3 compared 5 pretrained, production-grade VLMs on real PAN/Aadhaar field extraction. Until now these were two isolated case studies, connected only by prose reasoning.

This notebook connects them directly: it takes the *exact same* Stage 2 architecture and training loop, and teaches it the extraction task instead of captioning, using **only real, manually-verified ground truth** (`poc/manual_ground_truth_cache/{pan,aadhaar}.json` -- 15 images each; no synthetic data anywhere, per practice-head direction).

**Why leave-one-out cross-validation (LOOCV):** 15 labeled images per document type is the entire trustworthy dataset -- Phase 2's other ~85 images per type only have model-generated pseudo-labels, not verified truth. A single train/test split off 15 examples would be too noisy to report (each held-out example swings the score by ~7 points). LOOCV trains 15 separate models per document type, each holding out exactly one image, so every image gets scored exactly once by a model that never saw it during training -- the standard approach at this sample size.

**Expected result, stated up front:** this is very likely to underperform the 5 pretrained POC models by a wide margin, and that's the point, not a failed experiment. Two reasons, both already established in the report's literature review: (1) `clip-vit-base-patch32` processes images at 224x224 into just 50 visual tokens -- almost certainly too low-resolution to resolve fine printed document text, exactly why Qwen2.5-VL's *native dynamic-resolution* encoder is called out as a key differentiator for document tasks; (2) each fold trains on only ~14 examples, orders of magnitude less than any realistic training set, so genuine document-reading generalization isn't a reasonable expectation -- a smoke test on a tiny subset confirmed the model can perfectly memorize its training targets (near-zero loss) but on a held-out image mostly regurgitates another training example's field values rather than reading the new image.

## 1. Setup

In [1]:
import json
import sys
import time
from pathlib import Path

import torch
import torch.nn as nn
from PIL import Image

from transformers import CLIPVisionModel, AutoImageProcessor, AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

sys.path.insert(0, "../poc")
from extraction_lib import normalize

torch.manual_seed(42)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)


Using device: mps


## 2. Configuration

Same vision encoder, LLM, and LoRA hyperparameters as `minimal_vlm_llava_stage2_lora.ipynb` -- this is deliberately the *same recipe*, applied to a new task via new training data and target text, not a different architecture. Only the epoch count changes: Stage 2 used 2 epochs over ~8K Flickr8k images; here, ~14 examples per fold need far more passes to fit at all (validated via a smoke test: near-zero loss and perfect target-format reproduction by ~200-300 epochs).

In [2]:
VISION_MODEL_ID = "openai/clip-vit-base-patch32"
LLM_MODEL_ID = "HuggingFaceTB/SmolLM2-135M"
STAGE1_PROJECTOR_PATH = "stage1_projector.pt"

MAX_TEXT_LEN = 64
NUM_EPOCHS = 300
LORA_LR = 2e-4
PROJECTOR_LR = 5e-5
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

DOC_CONFIG = {
    "pan": {
        "gt_path": Path("../poc/manual_ground_truth_cache/pan.json"),
        "image_dir": Path("../data/PAN_Kaggle/train/images"),
        "field_labels": [("name", "NAME"), ("father_name", "FATHER"), ("dob", "DOB"), ("pan_number", "PAN")],
    },
    "aadhaar": {
        "gt_path": Path("../poc/manual_ground_truth_cache/aadhaar.json"),
        "image_dir": Path("../data/Adhaar_Kaggle/train/images"),
        "field_labels": [("name", "NAME"), ("dob", "DOB"), ("gender", "GENDER"), ("aadhaar_number", "AADHAAR")],
    },
}

RESULTS_FILE = "document_extraction_finetune_results.json"


## 3. Target text format (not JSON, deliberately)

A flat, delimited string -- e.g. `NAME: X | FATHER: Y | DOB: Z | PAN: W` -- rather than JSON. A non-instruction-tuned 135M base model, trained on a dozen examples, has a much more realistic chance of learning to produce this than precise JSON bracket/quote/comma syntax.

In [3]:
def build_target(doc_type, fields):
    labels = DOC_CONFIG[doc_type]["field_labels"]
    return " | ".join(f"{label}: {fields.get(key, '')}" for key, label in labels)


def parse_target(doc_type, text):
    labels = DOC_CONFIG[doc_type]["field_labels"]
    result = {key: "" for key, _ in labels}
    label_to_key = {label: key for key, label in labels}
    for seg in text.split("|"):
        seg = seg.strip()
        if ":" not in seg:
            continue
        label, value = seg.split(":", 1)
        label = label.strip().upper()
        if label in label_to_key:
            result[label_to_key[label]] = value.strip()
    return result


# Round-trip sanity check.
_sample = {"name": "PITABAS JANA", "father_name": "SHASHI JANA", "dob": "01/01/1965", "pan_number": "BUFPJ8690B"}
assert parse_target("pan", build_target("pan", _sample)) == _sample
print("Target format round-trips correctly.")


Target format round-trips correctly.


## 4. The model (identical to Stage 2's `MiniLLaVAStage2`, adapted to take precomputed vision features)

Since the vision encoder is frozen and never changes across epochs or folds, its output for each of the (tiny) set of real images is computed **once** and cached -- training then only has to run the trainable projector + LoRA-adapted LLM each step, not the full CLIP forward pass. Mathematically identical to calling the frozen vision encoder every step; just faster given ~30 fold-retrainings ahead.

In [4]:
class MiniLLaVADocExtract(nn.Module):
    def __init__(self, vision_hidden_size, llm, tokenizer):
        super().__init__()
        self.llm = llm
        self.tokenizer = tokenizer
        self.projector = nn.Linear(vision_hidden_size, llm.config.hidden_size)

    def _visual_tokens(self, vision_features):
        return self.projector(vision_features)

    def forward(self, vision_features, input_ids, attention_mask):
        visual_tokens = self._visual_tokens(vision_features)
        text_embeds = self.llm.get_input_embeddings()(input_ids)
        inputs_embeds = torch.cat([visual_tokens, text_embeds], dim=1)

        V = visual_tokens.shape[1]
        full_attention_mask = torch.cat(
            [torch.ones(vision_features.shape[0], V, dtype=torch.long, device=vision_features.device), attention_mask],
            dim=1,
        )
        text_labels = input_ids.clone()
        text_labels[attention_mask == 0] = -100
        visual_labels = torch.full((vision_features.shape[0], V), -100, dtype=torch.long, device=vision_features.device)
        labels = torch.cat([visual_labels, text_labels], dim=1)

        outputs = self.llm(inputs_embeds=inputs_embeds, attention_mask=full_attention_mask, labels=labels)
        return outputs.logits, outputs.loss

    @torch.no_grad()
    def generate(self, vision_features, max_new_tokens=MAX_TEXT_LEN):
        self.eval()
        visual_tokens = self._visual_tokens(vision_features)
        attention_mask = torch.ones(visual_tokens.shape[:2], dtype=torch.long, device=vision_features.device)
        return self.llm.generate(
            inputs_embeds=visual_tokens,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )


## 5. Load the frozen vision encoder and tokenizer once

In [5]:
image_processor = AutoImageProcessor.from_pretrained(VISION_MODEL_ID, use_fast=True)
vision_encoder = CLIPVisionModel.from_pretrained(VISION_MODEL_ID, dtype=torch.float32).to(device)
vision_encoder.requires_grad_(False)
vision_encoder.eval()
vision_hidden_size = vision_encoder.config.hidden_size

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Vision encoder + tokenizer loaded.")


[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_at

Vision encoder + tokenizer loaded.


## 6. LOOCV fine-tuning + evaluation (one function, run once per document type)

For each document type: precompute cached vision features for all labeled images, then for each held-out image, train a fresh model (fresh LoRA init, projector re-warm-started from Stage 1) on the other N-1, generate on the held-out image, and record the parsed prediction. `git status`-safe: the loop below never prints or returns individual raw field values outside the notebook's own live cell output (interactive display in-notebook is fine per project convention) -- only the aggregate scoring in Section 7 gets saved to disk.

In [6]:
def run_loocv(doc_type):
    config = DOC_CONFIG[doc_type]
    ground_truth = json.loads(config["gt_path"].read_text(encoding="utf-8"))
    names = list(ground_truth.keys())
    targets_by_name = {n: build_target(doc_type, f) for n, f in ground_truth.items()}

    # Cache frozen vision features once per image -- identical across every fold.
    vision_feat_cache = {}
    with torch.no_grad():
        for name in names:
            img = Image.open(config["image_dir"] / name).convert("RGB")
            pv = image_processor(images=img, return_tensors="pt")["pixel_values"][0].unsqueeze(0).to(device)
            vision_feat_cache[name] = vision_encoder(pixel_values=pv).last_hidden_state[0].cpu()

    predictions = {}
    t0 = time.time()
    for i, held_out in enumerate(names, 1):
        train_names = [n for n in names if n != held_out]

        base_llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL_ID, dtype=torch.float32).to(device)
        base_llm.config.pad_token_id = tokenizer.pad_token_id
        lora_config = LoraConfig(task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA,
                                  lora_dropout=LORA_DROPOUT, target_modules=LORA_TARGET_MODULES)
        llm = get_peft_model(base_llm, lora_config)

        model = MiniLLaVADocExtract(vision_hidden_size, llm, tokenizer).to(device)
        model.projector.load_state_dict(torch.load(STAGE1_PROJECTOR_PATH, map_location=device))

        optimizer = torch.optim.AdamW([
            {"params": model.projector.parameters(), "lr": PROJECTOR_LR},
            {"params": [p for n, p in model.llm.named_parameters() if p.requires_grad], "lr": LORA_LR},
        ])

        # Full-batch training -- N-1 examples is tiny enough to fit as one batch per step.
        ids_list, attn_list, feat_list = [], [], []
        for name in train_names:
            target = targets_by_name[name]
            ids = tokenizer(target, add_special_tokens=False)["input_ids"]
            ids = ids[: MAX_TEXT_LEN - 1] + [tokenizer.eos_token_id]
            attn = [1] * len(ids)
            pad_len = MAX_TEXT_LEN - len(ids)
            ids = ids + [tokenizer.pad_token_id] * pad_len
            attn = attn + [0] * pad_len
            ids_list.append(ids)
            attn_list.append(attn)
            feat_list.append(vision_feat_cache[name])
        train_vf = torch.stack(feat_list).to(device)
        train_ids = torch.tensor(ids_list, dtype=torch.long).to(device)
        train_attn = torch.tensor(attn_list, dtype=torch.long).to(device)

        model.train()
        for epoch in range(NUM_EPOCHS):
            _, loss = model(train_vf, train_ids, train_attn)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        held_vf = vision_feat_cache[held_out].unsqueeze(0).to(device)
        gen_ids = model.generate(held_vf)
        gen_text = tokenizer.decode(gen_ids[0], skip_special_tokens=True)
        predictions[held_out] = parse_target(doc_type, gen_text)

        elapsed = time.time() - t0
        print(f"[{doc_type}] fold {i}/{len(names)} done (final train loss {loss.item():.4f}), "
              f"elapsed {elapsed/60:.1f} min")

    return ground_truth, predictions


## 7. Run LOOCV for PAN, then Aadhaar, and score against ground truth

In [7]:
def score(doc_type, ground_truth, predictions):
    field_names = [key for key, _ in DOC_CONFIG[doc_type]["field_labels"]]
    per_field_correct = {f: 0 for f in field_names}
    document_correct = 0
    for name, truth in ground_truth.items():
        pred = predictions[name]
        matches = {f: normalize(pred.get(f, "")) == normalize(truth.get(f, "")) for f in field_names}
        for f, ok in matches.items():
            if ok:
                per_field_correct[f] += 1
        if all(matches.values()):
            document_correct += 1
    n = len(ground_truth)
    return {
        "n": n,
        "per_field_accuracy": {f: per_field_correct[f] / n for f in field_names},
        "document_accuracy": document_correct / n,
    }


results = {}
for doc_type in ["pan", "aadhaar"]:
    print(f"\n=== Running LOOCV for {doc_type} ===")
    ground_truth, predictions = run_loocv(doc_type)
    results[doc_type] = score(doc_type, ground_truth, predictions)
    acc = results[doc_type]["document_accuracy"] * 100
    n = results[doc_type]["n"]
    print(f"{doc_type}: full-document accuracy {acc:.1f}% (n={n})")



=== Running LOOCV for pan ===


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

[pan] fold 1/15 done (final train loss 0.0006), elapsed 2.4 min


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

[pan] fold 2/15 done (final train loss 0.0006), elapsed 37.1 min


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

... (51 outputs left out)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

[aadhaar] fold 14/15 done (final train loss 0.0004), elapsed 36.8 min


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

[aadhaar] fold 15/15 done (final train loss 0.0004), elapsed 39.2 min
aadhaar: full-document accuracy 0.0% (n=15)


## 8. Save aggregate-only results

In [8]:
summary = {
    "model": "MiniLLaVA (Stage 2 recipe: clip-vit-base-patch32 + LoRA-adapted SmolLM2-135M, LOOCV fine-tuned on real ground truth)",
    "num_epochs_per_fold": NUM_EPOCHS,
    "results": results,
}

with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"Saved aggregate-only results to {RESULTS_FILE}")
print(json.dumps(results, indent=2))


Saved aggregate-only results to document_extraction_finetune_results.json
{
  "pan": {
    "n": 15,
    "per_field_accuracy": {
      "name": 0.0,
      "father_name": 0.0,
      "dob": 0.0,
      "pan_number": 0.0
    },
    "document_accuracy": 0.0
  },
  "aadhaar": {
    "n": 15,
    "per_field_accuracy": {
      "name": 0.0,
      "dob": 0.0,
      "gender": 0.4,
      "aadhaar_number": 0.0
    },
    "document_accuracy": 0.0
  }
}


## 9. Notes for the report

- **This is the direct, empirical connection between Week 2 and Week 3** the report's Synthesis section previously argued only from literature-based reasoning: the exact architecture and training recipe used successfully for Flickr8k captioning, retrained on real document extraction with real ground truth, to see how far the same recipe generalizes to a new task and a much harder input domain.
- **Compare directly against the 5 POC models' "GT accuracy" column** in `poc/model_comparison_summary.json` (same n=15 per doc type, same normalized exact-match methodology, same images) -- this result is a legitimate 6th row in that table, not a separate, incomparable number.
- **Whatever the numbers show, report them plainly** -- a large gap in either direction is informative. If MiniLLaVA badly underperforms (the expected outcome, per Section 0's reasoning), that's direct empirical evidence for why architecture/training-data-specificity (not just parameter count) explains the pretrained models' advantage. If it's closer than expected, that's worth investigating and reporting too.